In [9]:
using LinearAlgebra
using BoundaryValueDiffEq
using Plots
using DifferentialEquations
using BSplineKit
using DelimitedFiles

In [10]:
function sol_baseflowODE(tspan,Num)

    function oneDiskODE!(du, u , p, t)
        
        U = u[1]
        dU = u[2]
        V = u[3]
        dV = u[4]
        W = u[5]
        du[1] = dU
        ddU = U^2 + W*dU - (V+1.0e0)^2
        du[2] = ddU
        ddV = 2.0e0*U*(V + 1.0e0) + W*dV
        du[3] = dV
        du[4] = ddV                          
        du[5] = -2.0e0*U

    end
    function oneDiskbc!(residual, u , p, t)

        residual[1] = u[begin][1] 
        residual[2] = u[begin][3] 
        residual[3] = u[begin][5] 
        residual[4] = u[end][1] 

        residual[5] = u[end][3] + 1.0e0
    end
        prob = BVProblem(oneDiskODE!, oneDiskbc!, [0.0, 0.5103341351120374, 0.0, -0.6151547026271073, 0.0] ,tspan, dtmax=0.01)
        sol = solve(prob, Shooting(Vern7()), dt=0.01)
        t=range(0.0, 20, Num)
        u=sol(t)
    
    return u , t

end

sol_baseflowODE (generic function with 1 method)

In [11]:
function velocity(u)

    U = u[1 , :]
    dU = u[2 , :]
    V = u[3 , :]
    dV = u[4 , :]
    W = u[5 , :]
    F_U = itp = interpolate(t, U , BSplineOrder(4))
    F_dU = itp = interpolate(t, dU , BSplineOrder(4))
    F_dV = itp = interpolate(t, dV , BSplineOrder(4))
    F_W= itp = interpolate(t, W , BSplineOrder(4))
    
    return U,dU,V,dV,W,F_U,F_dU,F_dV,F_W

end

velocity (generic function with 1 method)

In [12]:
function T_start(F_dU,F_dV,F_W,sigma,gamma,Tw,tspan,Num)
    gamma = gamma
    Tw = Tw
    sigma = sigma
    function ODE_T!(du,u,p,t)
        T = u[1]
        dT = u[2]
        du[1] = dT
        du[2] = (F_W(t) * u[2] )*sigma
    end
    function bc2!(residual, u, p, t)
        residual[1] = u[begin][1] - Tw
        residual[2] = u[end][1] - 1
    end
        prob = BVProblem(ODE_T! , bc2! , [0,0] , tspan)
        sol = solve(prob , Shooting(Vern7()) , dt = 0.001)
        t=range(0.0, 20, Num)
        T=sol(t)
        dT = T[2,:]
        T = T[1,:]
    return T , dT

end

T_start (generic function with 1 method)

In [13]:
function Cheb(u,v,w,T,t,N)
    θ = range(0,length=N+1,stop=pi)
    x = reshape(-cos.(θ), N+1, 1)
    c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
    X = repeat(x, 1, N+1);
    dX = X - X';
    D = (c * (1 ./ c)') ./ (dX .+ I(N+1));
    D = D - diagm(vec(sum(D, dims=2))); 
    for i=1:N+1
        x[i] = 10 * x[i] .+ 10
    end
    D = 0.1 * D
    # for i=1:N+1
    #     D[i,:]=D[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))
    # end
    # for i=1:N+1
    #     x[i]=(4*x[i]^3-2*x[i]^2+6*x[i]+12)/(-2*x[i]^3+x[i]^2-3*x[i]+4)
    #     if x[i]>15
    #         x[i]=15
    #     end
    # end
    itpw = itp = interpolate(t, w , BSplineOrder(4))
    itpu = itp = interpolate(t, u , BSplineOrder(4))
    itpv = itp = interpolate(t, v , BSplineOrder(4))
    itpT = itp = interpolate(t, T , BSplineOrder(4))
    u = zeros(N+1,1)
    v = zeros(N+1,1)
    w = zeros(N+1,1)
    T = zeros(N+1,1)
    for i=1:N+1
        u[i,1] = itpu(x[i])
        v[i,1] = itpv(x[i])
        w[i,1] = itpw(x[i])
        T[i,1] = itpT(x[i])
    end
    return D,x,u,v,w,T
end

Cheb (generic function with 1 method)

In [14]:
function y_DiffMat(y_range,N)
    y0 = minimum(y_range)
    y1 = maximum(y_range)
    dely = (y1 - y0) / (N-1)
    D = zeros(N,N)
    D2 = zeros(N,N)
    for i = 1:N
        if i == 1
            D[i,i] = -3 / (2*dely)
            D[i,i+1] = 4 / (2*dely)
            D[i,i+2] = -1 / (2*dely)
            D2[i,i] = 2 / (dely^2)
            D2[i,i+1] = -5 / (dely^2)
            D2[i,i+2] = 4 / (dely^2)
            D2[i,i+3] = -1 /(dely^2)
        elseif i == N
            D[i,i] = 3 / (2*dely)
            D[i,i-1] = -4 / (2*dely)
            D[i,i-2] = 1 / (2*dely)
            D2[i,i] = 2 / (dely^2)
            D2[i,i-1] = -5 / (dely^2)
            D2[i,i-2] = 4 / (dely^2)
            D2[i,i-3] = -1 /(dely^2)
        elseif i !== 1;N
            D[i,i+1] = 1 / (2*dely)
            D[i,i-1] = -1 / (2*dely)
            D2[i,i-1] = 1 / (dely^2)
            D2[i,i] = -2 / (dely^2)
            D2[i,i+1] = 1 / (dely^2)
        end
    end

    return D,D2
end

y_DiffMat (generic function with 1 method)

In [15]:
function var!(x,x1)
    a = length(x)
    b = x1 .- x
    d = 0
    for i = 1 : a
        c = b[i,1]^2
        d = c + d 
    end
    e = sqrt(d) / a
    return e
end

var! (generic function with 1 method)

In [16]:
tspan = (0,20)
Num = 10001
t = range(0,20,Num )
sigma = 0.7
Tw = 0.5
gamma = 1.4
N = 199
u,z = sol_baseflowODE(tspan,Num)
u0,du0,v0,dv0,w0,F_u,F_du,F_dv,F_w = velocity(u)
T0,dT0 = T_start(F_du,F_dv,F_w,sigma,gamma,Tw,tspan,Num)
D,x,u0,v0,w0,T0 = Cheb(u0,v0,w0,T0,t,N)
D2 = D^2
d2u0 = D2 * u0
du0 = D * u0
d2v0 = D2 * v0
dv0 = D * v0
T_temp = T0

200×1 Matrix{Float64}:
 0.5
 0.5002013224952283
 0.5008052398043646
 0.5018116013516054
 0.5032201557957625
 0.5050305496651035
 0.5072423239035945
 0.5098549068822307
 0.5128676021495463
 0.5162795690093736
 ⋮
 0.9999957872263154
 0.999995827943686
 0.9999958628740244
 0.9999958921730785
 0.9999959159708571
 0.9999959343725309
 0.9999959474591578
 0.999995955288242
 0.9999959578941295

In [18]:
data = [x u0 v0 w0 T0]

200×5 Matrix{Float64}:
  0.0         -2.5078e-27    0.0           0.0          0.5
  0.0012461    0.000635027  -0.000767503  -7.91634e-7   0.500201
  0.00498411   0.00253066   -0.0030698    -1.26337e-5   0.500805
  0.0112131    0.0056587    -0.00690614   -6.36848e-5   0.501812
  0.0199315    0.00997267   -0.0122749    -0.000200074  0.50322
  0.0311371    0.0154086    -0.019173     -0.000484714  0.505031
  0.0448272    0.0218859    -0.0275952    -0.000995686  0.507242
  0.0609983    0.0293091    -0.0375331    -0.00182423   0.509855
  0.0796464    0.0375689    -0.0489743    -0.00307236   0.512868
  0.100767     0.0465443    -0.0619014    -0.00485027   0.51628
  ⋮                                                     
 19.9204       1.4024e-9    -1.0          -0.884473     0.999996
 19.939        1.06514e-9   -1.0          -0.884473     0.999996
 19.9552       7.77137e-10  -1.0          -0.884473     0.999996
 19.9689       5.36525e-10  -1.0          -0.884473     0.999996
 19.9801       3.

In [19]:
writedlm("start.dat",data,'\t')

In [432]:
Ma = 6
x_i = 0.0001
delx = 0.0001
Mx = Ma * x_i
j=1
# for j = 1 : 2
    # x_i = j * delx
    # Mx = Ma * x_i
    for i = 1
        dT_temp = D * T_temp 
        d2T_temp = D2 * T_temp
        F = 1 * (((Mx^2 * (gamma - 1)) .* (du0[2:end-1,1] .* du0[2:end-1,1] + dv0[2:end-1,1] .* dv0[2:end-1,1] )) .- w0[2:end-1,1] .* dT_temp[2:end-1,1] .+ (1/sigma) .* d2T_temp[2:end-1,1])
        E = u0[2 : end-1,1] .* I(N-1)
        T_temp = E\F .+ T_temp[2:end-1,1]
        insert!(T_temp,1,Tw)
        insert!(T_temp,N+1,1)
        for i = 2 : 1 : N-1 
            if abs(T_temp[i,1] - T_temp[i-1,1]) < 0.001 && 0.999<T_temp[i,1]<1.001
                T_temp[i:end,1] .= T_temp[i,1]
            end
        end
    end
# end

In [ ]:
dT_temp = D * T_temp 
d2T_temp = D2 * T_temp
F = 1 * (((Mx^2 * (gamma - 1)) .* (du0[2:end-1,1] .* du0[2:end-1,1] + dv0[2:end-1,1] .* dv0[2:end-1,1] )) .- w0[2:end-1,1] .* dT_temp[2:end-1,1] .+ (1/sigma) .* d2T_temp[2:end-1,1])
E = u0[2 : end-1,1] .* I(N-1)
T_temp = gmres(E,F) .+ T_temp[2:end-1,1]
insert!(T_temp,1,Tw)
insert!(T_temp,N+1,1)

In [ ]:
dT_temp = D * T_temp 

In [ ]:
D2 * T_temp

In [ ]:
T_temp

In [ ]:
plot(x,T_temp)